# Inspección de `smoke_local_scraper.db`

Este notebook ayuda a explorar la base de datos generada por `tests/smoke_local_scraper.py`.

Incluye:
- conexión a SQLite
- listado de tablas y esquema
- filas de muestra
- análisis de columnas y nulos
- métricas básicas
- consultas de exploración
- exportación de resultados

## 1. Importar librerías y configurar la conexión

Importamos `sqlite3`, `pandas` y algunas utilidades para explorar la base desde el notebook.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
from IPython.display import display, Markdown

DB_PATH = Path("smoke_local_scraper.db")
if not DB_PATH.exists():
    raise FileNotFoundError(f"No se encuentra la base de datos: {DB_PATH.resolve()}")

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
print(f"Conectado a: {DB_PATH.resolve()}")

## 2. Conectarse a `smoke_local_scraper.db`

Abrimos la base SQLite del repositorio y dejamos una conexión reutilizable.

### Comprobación rápida de existencia

Si el archivo no existe, el notebook falla al instante con un mensaje claro.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
from IPython.display import display, Markdown

DB_PATH = Path("smoke_local_scraper.db")
if not DB_PATH.exists():
    raise FileNotFoundError(f"No se encuentra la base de datos: {DB_PATH.resolve()}")

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
print(f"Conectado a: {DB_PATH.resolve()}")

## 3. Listar tablas y esquema de la base de datos

Consultamos `sqlite_master` para ver las tablas disponibles y las sentencias `CREATE TABLE`.

In [ ]:
tables_df = pd.read_sql_query(
    "SELECT name AS table_name, sql AS create_sql FROM sqlite_master WHERE type='table' ORDER BY name",
    conn,
)
display(tables_df)

for _, row in tables_df.iterrows():
    print(f"\n### {row['table_name']}")
    print(row['create_sql'])

## 4. Inspeccionar filas de muestra por tabla

Leemos unas pocas filas de cada tabla para ver el contenido real.

In [ ]:
for table_name in tables_df['table_name']:
    print(f"\n### {table_name}")
    sample_df = pd.read_sql_query(f"SELECT * FROM {table_name} LIMIT 30", conn)
    display(sample_df)

## 5. Analizar columnas, tipos y valores nulos

Creamos un resumen por tabla con tipos y proporción de nulos.

In [ ]:
def inspect_table(table_name: str) -> pd.DataFrame:
    columns_df = pd.read_sql_query(f"PRAGMA table_info({table_name})", conn)
    row_count = pd.read_sql_query(f"SELECT COUNT(*) AS row_count FROM {table_name}", conn).iloc[0]["row_count"]
    if row_count == 0:
        columns_df["null_count"] = 0
        columns_df["null_pct"] = 0.0
        return columns_df

    null_exprs = ", ".join(
        [f"SUM(CASE WHEN {col} IS NULL OR TRIM(CAST({col} AS TEXT)) = '' THEN 1 ELSE 0 END) AS {col}" for col in columns_df['name']]
    )
    null_counts = pd.read_sql_query(f"SELECT {null_exprs} FROM {table_name}", conn).iloc[0].to_dict()
    columns_df["null_count"] = columns_df["name"].map(null_counts).fillna(0).astype(int)
    columns_df["null_pct"] = (columns_df["null_count"] / row_count).round(3)
    return columns_df

for table_name in tables_df['table_name']:
    print(f"\n### {table_name}")
    display(inspect_table(table_name))

## 6. Calcular conteos y métricas básicas

Calculamos tamaño, unicidad y algunas estadísticas rápidas para la tabla principal.

In [ ]:
properties_df = pd.read_sql_query("SELECT * FROM properties", conn)
summary = pd.DataFrame(
    {
        "metric": [
            "rows",
            "columns",
            "unique_property_id",
            "active_rows",
            "inactive_rows",
            "price_min",
            "price_median",
            "price_max",
            "rooms_filled",
            "bathrooms_filled",
            "sqm_filled",
        ],
        "value": [
            len(properties_df),
            properties_df.shape[1],
            properties_df["property_id"].nunique(dropna=True),
            (properties_df["status"] == "active").sum() if "status" in properties_df else None,
            (properties_df["status"] == "inactive").sum() if "status" in properties_df else None,
            properties_df["price"].min(),
            properties_df["price"].median(),
            properties_df["price"].max(),
            properties_df["rooms"].notna().sum(),
            properties_df["bathrooms"].notna().sum(),
            properties_df["sqm"].notna().sum(),
        ],
    }
)
display(summary)
display(properties_df.describe(include="all"))

## 7. Ejecutar consultas de exploración y filtrado

Consultas SQL útiles para ver datos activos, detectar duplicados o encontrar registros sospechosos.

In [ ]:
available_tables = set(tables_df['table_name'])

queries = {
    'active_properties': """
        SELECT property_id, source, title, price, rooms, bathrooms, sqm, city, status
        FROM properties
        WHERE status = 'active'
        ORDER BY title
    """,
    'missing_required_fields': """
        SELECT property_id, title, price, rooms, bathrooms, sqm
        FROM properties
        WHERE title IS NULL OR price IS NULL OR rooms IS NULL OR bathrooms IS NULL OR sqm IS NULL
        ORDER BY title
    """,
}

if 'price_history' in available_tables:
    price_history_columns = pd.read_sql_query("PRAGMA table_info(price_history)", conn)['name'].tolist()
    timestamp_candidates = [
        'seen_at', 'scraped_at', 'created_at', 'updated_at', 'timestamp', 'date', 'captured_at'
    ]
    order_col = next((col for col in timestamp_candidates if col in price_history_columns), None)

    selected_cols = ['property_id', 'price']
    if order_col:
        selected_cols.append(order_col)

    order_clause = f"ORDER BY {order_col} DESC" if order_col else ""
    queries['price_history'] = f"""
        SELECT {', '.join(selected_cols)}
        FROM price_history
        {order_clause}
    """

for name, sql in queries.items():
    print(f"\n### {name}")
    try:
        display(pd.read_sql_query(sql, conn))
    except Exception as exc:
        print(f"Error ejecutando {name}: {exc}")

## 8. Exportar resultados de inspección

Guardamos CSVs con resúmenes y tablas de salida para analizarlos fuera del notebook.

In [ ]:
export_dir = Path('notebook_exports')
export_dir.mkdir(exist_ok=True)

tables_df.to_csv(export_dir / 'tables.csv', index=False)
summary.to_csv(export_dir / 'summary.csv', index=False)
properties_df.to_csv(export_dir / 'properties.csv', index=False)

print(f'Exported CSV files to: {export_dir.resolve()}')

In [ ]:
# Verificar columna de alta inicial y calcular antigüedad de anuncios
properties_columns = pd.read_sql_query("PRAGMA table_info(properties)", conn)['name'].tolist()
first_seen_candidates = ['first_seen', 'first_seen_at', 'created_at', 'published_at', 'date']
first_seen_col = next((col for col in first_seen_candidates if col in properties_columns), None)

if not first_seen_col:
    print("No se encontró una columna de primera publicación en properties.")
else:
    print(f"Usando columna de primera publicación: {first_seen_col}")
    sql_first_seen = f"""
        SELECT
            property_id,
            title,
            {first_seen_col} AS first_seen,
            CAST(julianday('now') - julianday({first_seen_col}) AS INTEGER) AS days_online
        FROM properties
        WHERE {first_seen_col} IS NOT NULL
        ORDER BY days_online DESC, first_seen ASC
        LIMIT 30
    """
    display(pd.read_sql_query(sql_first_seen, conn))

## 9. Anuncios activos más antiguos

Esta vista muestra los anuncios que siguen activos y llevan más tiempo publicados, combinando `first_seen` y `last_seen`.

In [ ]:
properties_columns = pd.read_sql_query("PRAGMA table_info(properties)", conn)['name'].tolist()
active_column = 'status' if 'status' in properties_columns else None

first_seen_candidates = ['first_seen', 'first_seen_at', 'created_at', 'published_at', 'date']
last_seen_candidates = ['last_seen', 'last_seen_at', 'updated_at', 'scraped_at', 'date']

first_seen_col = next((col for col in first_seen_candidates if col in properties_columns), None)
last_seen_col = next((col for col in last_seen_candidates if col in properties_columns), None)

if not first_seen_col and not last_seen_col:
    print('No se encontraron columnas útiles de seguimiento temporal en properties.')
else:
    print(f'Usando first_seen={first_seen_col or "n/a"} y last_seen={last_seen_col or "n/a"}')
    select_cols = [
        'property_id',
        'title',
        'source',
        'city',
        'price',
        'rooms',
        'bathrooms',
        'sqm',
    ]
    if active_column:
        select_cols.append('status')
    if first_seen_col:
        select_cols.append(f'{first_seen_col} AS first_seen')
        select_cols.append("CAST(julianday('now') - julianday(first_seen) AS INTEGER) AS days_online")
    else:
        select_cols.append('NULL AS first_seen')
        select_cols.append('NULL AS days_online')
    if last_seen_col:
        if last_seen_col == first_seen_col:
            select_cols.append(f'{last_seen_col} AS last_seen')
        else:
            select_cols.append(f'{last_seen_col} AS last_seen')
        select_cols.append("CAST(julianday('now') - julianday(last_seen) AS INTEGER) AS days_since_last_seen")
    else:
        select_cols.append('NULL AS last_seen')
        select_cols.append('NULL AS days_since_last_seen')

    where_clauses = []
    if active_column:
        where_clauses.append("status = 'active'")
    if first_seen_col:
        where_clauses.append(f'{first_seen_col} IS NOT NULL')
    if last_seen_col:
        where_clauses.append(f'{last_seen_col} IS NOT NULL')

    where_sql = ''
    if where_clauses:
        where_sql = 'WHERE ' + ' AND '.join(where_clauses)

    sql_oldest_active = f"""
        SELECT {', '.join(select_cols)}
        FROM properties
        {where_sql}
        ORDER BY days_online DESC, days_since_last_seen DESC, first_seen ASC, title ASC
        LIMIT 25
    """

    oldest_active_df = pd.read_sql_query(sql_oldest_active, conn)
    summary_columns = [
        col for col in [
            'property_id',
            'title',
            'first_seen',
            'days_online',
            'price',
        ]
        if col in oldest_active_df.columns
    ]
    display(oldest_active_df[summary_columns].head(10))

## 10. Precio inicial vs precio actual

Esta vista compara el precio de la primera vez que se vio el anuncio con el precio actual guardado en la tabla.

In [ ]:
properties_columns = pd.read_sql_query("PRAGMA table_info(properties)", conn)['name'].tolist()
price_first_seen_col = 'price_first_seen' if 'price_first_seen' in properties_columns else None

if not price_first_seen_col:
    print('No se encontró la columna price_first_seen en properties.')
else:
    sql_price_evolution = """
        SELECT
            property_id,
            title,
            source,
            city,
            price_first_seen,
            price,
            CASE
                WHEN price_first_seen IS NOT NULL AND price IS NOT NULL
                THEN price - price_first_seen
                ELSE NULL
            END AS price_delta,
            CASE
                WHEN price_first_seen IS NOT NULL AND price IS NOT NULL AND price_first_seen != 0
                THEN ROUND((price - price_first_seen) * 100.0 / price_first_seen, 2)
                ELSE NULL
            END AS price_delta_pct
        FROM properties
        WHERE status = 'active'
          AND price_first_seen IS NOT NULL
        ORDER BY ABS(COALESCE(price - price_first_seen, 0)) DESC, title ASC
        LIMIT 25
    """
    price_evolution_df = pd.read_sql_query(sql_price_evolution, conn)
    display(price_evolution_df[[
        'property_id',
        'title',
        'price_first_seen',
        'price',
        'price_delta',
        'price_delta_pct',
    ]].head(10))